# Week 1: Introduction to Deep Learning - Homework

**ML2: Advanced Machine Learning**

**Estimated Time**: 1 hour

---

This homework combines programming exercises and knowledge-based questions to reinforce this week's concepts.

## Setup

Run this cell to import necessary libraries:

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print('✓ Libraries imported successfully')

✓ Libraries imported successfully


---
## Part 1: Programming Exercises (60%)

Complete the following programming tasks. Read each description carefully and implement the requested functionality.

### Exercise 1: Experiment: Observing Feature Learning

**Time**: 8 min

Run this code to visualize what happens when a network learns features automatically vs. using hand-crafted features. Observe the outputs and answer the reflection questions below.

In [7]:
import torch
import torch.nn as nn
import numpy as np

# Simulate a simple pattern recognition task
# Pattern: Detect if sum of inputs > 5
np.random.seed(42)
torch.manual_seed(42)

# Generate data
X = torch.randn(100, 4)  # 100 samples, 4 features
y = (X.sum(dim=1) > 0).float()  # Label: 1 if sum > 0, else 0

# Network that LEARNS features
model = nn.Sequential(
    nn.Linear(4, 8),   # Learned feature extraction
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

# Train for a few steps
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.BCELoss()

for epoch in range(20):
    optimizer.zero_grad()
    predictions = model(X).squeeze()
    loss = loss_fn(predictions, y)
    loss.backward()
    optimizer.step()

print(f"Final loss: {loss.item():.4f}")
print(f"First layer weights (learned features):")
print(model[0].weight.data)

# TODO: After running, answer reflection questions below

Final loss: 0.6108
First layer weights (learned features):
tensor([[-0.1617,  0.3287,  0.0248,  0.0042],
        [-0.0987, -0.4408, -0.4484, -0.2179],
        [-0.2584,  0.1583, -0.1498, -0.1987],
        [ 0.5144,  0.2492,  0.4296, -0.1357],
        [-0.1449,  0.1902, -0.2641, -0.2845],
        [-0.0614, -0.2262, -0.4333,  0.2482],
        [ 0.3795,  0.0395, -0.1429, -0.5230],
        [-0.0965,  0.1722,  0.2787,  0.0054]])


### Exercise 2: Experiment: Network Without Nonlinearity

**Time**: 10 min

This experiment demonstrates why activation functions are essential. Compare two networks: one with ReLU, one without.

In [ ]:
import torch
import torch.nn as nn

# Network WITH nonlinearity (ReLU)
network_with_relu = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 15),
    nn.ReLU(),
    nn.Linear(15, 5)
)

# Network WITHOUT nonlinearity (just linear layers)
network_without_relu = nn.Sequential(
    nn.Linear(10, 20),
    nn.Linear(20, 15),
    nn.Linear(15, 5)
)

# Test input
x = torch.randn(1, 10)

# Compare outputs
output_with = network_with_relu(x)
output_without = network_without_relu(x)

print("With ReLU output:", output_with)
print("Without ReLU output:", output_without)

# TODO: Now manually compute what network_without_relu is equivalent to
# Hint: Multiple linear transformations collapse into a single linear transformation
# Can you express the 3-layer linear network as a SINGLE equivalent linear layer?

# Apparently it doesnt work like this lulz
#man_no_relu_p1 = [10, 20] * [20, 15]
#man_no_relu_ans = man_no_relu_p1 * [15, 5]
#print(man_no_relu_ans)

# So since the variable network_without_relu already has the nn.linear(x,x) set up in a equece, I guess I can use that
# I need to extract the weights
#layer 1: (10 in, 20 out)
w1 = network_without_relu[0].weight
b1 = network_without_relu[0].bias

#layer 2: (20 in, 15 out)
w2 = network_without_relu[1].weight
b2 = network_without_relu[1].bias

#layer 3: (15 in, 5 out)
w3 = network_without_relu[2].weight
b3 = network_without_relu[2].bias

#im curious what it looks like
print("Layer 1")
print(w1)
print(b1)
print("layer 2")
print(w2)
print(b2)
print("layer 3")
print(w3)
print(b3)

#Create each part of the queation to start ptting it into a single linear layer
step1_output = (x @ w1.T) + b1 #I guess @ multiplies matrices
step2_output = (step1_output @ w2.T) + b2 
finalstep_output = (step2_output @ w3.T) + b3

#compare it with the thing above
print("OG NN linear output:", output_without)
print("My manual mishposh :", finalstep_output) # (ﾉ◕ヮ◕)ﾉ*:･ﾟ✧ it matches!!!

# It still is three layers, ya just gotta collapse it into one layer
collapsew = w3 @ w2 @ w1
collapseb = b3 + (b2 @ w3.T) + (b1 @ w2.T @ w3.T) # a bit more confusing than I thought
collapsed_output = (x @ collapsew.T) + collapseb

print("OG NN linear output:", output_without)
print("My manual mishposh :", finalstep_output) 
print("Collapsed Output:", collapsed_output)

#it a roundaabout, but **yes**. You can collapse a nn without relu into a single layer.

With ReLU output: tensor([[ 0.0194, -0.0448, -0.1004, -0.0552, -0.1359]],
       grad_fn=<AddmmBackward0>)
Without ReLU output: tensor([[ 0.1235, -0.0809,  0.2334, -0.3174,  0.2876]],
       grad_fn=<AddmmBackward0>)
Layer 1
Parameter containing:
tensor([[ 0.2097,  0.1614,  0.0639, -0.3094,  0.0281,  0.0472,  0.0896,  0.3117,
          0.2416,  0.1042],
        [ 0.2881, -0.0205,  0.2045,  0.0202,  0.0636,  0.0807, -0.0603,  0.3005,
          0.2551, -0.1496],
        [-0.2375,  0.2817,  0.1269, -0.1311, -0.3120, -0.0239,  0.1090,  0.2846,
          0.1361,  0.0253],
        [ 0.2799,  0.0963,  0.1053, -0.0500,  0.2855,  0.1324, -0.0401,  0.1094,
         -0.1963, -0.2742],
        [ 0.3136,  0.1730,  0.0851, -0.2200,  0.2827, -0.0326,  0.0741, -0.1923,
         -0.0619,  0.0696],
        [-0.1354,  0.0361, -0.2136, -0.1496, -0.2586,  0.2692,  0.0231,  0.0472,
          0.1894,  0.2969],
        [ 0.2532,  0.1899, -0.0186,  0.1219,  0.1501,  0.1733, -0.2403, -0.2466,
         -0.3091, 

---
## Part 2: Knowledge Questions (40%)

Answer the following questions to test your conceptual understanding.

### Question 1 (Short Answer)

**Question 1 - Automatic Feature Learning (Conceptual)**

Traditional machine learning for image classification requires manually designing features (e.g., edge detectors, color histograms, texture filters). Deep learning does not.

Explain in 3-4 sentences:
1. WHY can deep networks learn features automatically?
2. WHAT enables this (what architectural property)?
3. What is the tradeoff (what does deep learning need more of)?

**Hint**: Think about what happens in each layer of a deep network and how backpropagation adjusts those layers.

**Your Answer**:

The reason why deep learning doesnt need a manual touich for designing features is because of both the ability to use backpropagation and it's layered approach. As you feed the data through the first layer, the network is able to start looking at the raw pixels and able to adust its own weights to detects patterns that are more important to the predicitn the response.  response. The loss function is able to help it determine the which exact pixels are important, which initiall helps the network start to form the outlines and basic lines of the response. Usually, in order to have an effective neural network, we need a masive amount of data and a decent amount of computational power. Especially compared to traditional statistics. 

### Question 2 (Short Answer)

**Question 2 - Feature Hierarchy (Conceptual)**

In a deep CNN for face recognition:
- Layer 1 might detect edges
- Layer 2 might detect facial features (eyes, nose)
- Layer 3 might detect whole faces

Explain: Why does depth create this hierarchy? What would happen if you used a single-layer network instead?

**Hint**: Consider how each layer builds on representations from the previous layer.

**Your Answer**:

So basically this layered approach is iterative learning. It uses the first layer to obtain a general understanding such as edges and outlines. Once it's learned those key features, we run a Relu and with what the network had learned we feed the next layer we feed the output from layer 1 to the the next layer, this layer would be generally focused on  on to detecting facial features. The process repeast and we fine tune the next layer. Eventually we're able to detect an holistic object like a face. At the end of the network, we would run it through a loss function and optimizer to improve through on the next epoch. If you had used a single layer network, there would be no "relu" which would effectively have the network try and learn the whole face all at once. Intuitively it may not seem and ineffective, but it's inefficient. Inputting a relu inbetween each layer, helps the network hone in on specific features iteretively. 

### Question 3 (Short Answer)

**Question 3 - Nonlinearity Experiment Reflection**

Based on the 'Network Without Nonlinearity' experiment above:

Prove mathematically or explain conceptually why the 3-layer network without ReLU is equivalent to a SINGLE linear layer. What does this tell you about the necessity of activation functions?

**Hint**: Remember: Linear(Linear(x)) = Linear(x) because you can multiply weight matrices together.

**Your Answer**:

Oh, Im gonna reference my earlier code. If you just multiply all the matrices from each nn.linear output it basically has the same output. It goes and basically tells you that without the ReLU, there is no interatively learning. Its like trying to get to the third floor of a building, but trying use your legs to jump a whole flight up rather than using smaller steps (layers + ReLU). 

### Question 4 (Multiple Choice)

**Question 4 - Understanding Nonlinearity**

A neural network with 10 layers but NO activation functions can represent:

A) Any possible function (universal approximation)
B) Only linear functions
C) Only polynomial functions
D) Only step functions

A) Any possible function (universal approximation)
B) Only linear functions
C) Only polynomial functions
D) Only step functions

**Hint**: What happens when you compose linear transformations?

**Your Answer**: It's B

**Explanation**: It goes over the last question, without the "ReLU" we are able to make it into a linear function by multiplying the matrices. 

### Question 5 (Short Answer)

**Question 5 - ReLU Design Choice**

ReLU(x) = max(0, x) is one of the simplest possible nonlinear functions. Yet it became the dominant activation function (replacing sigmoid).

Explain TWO advantages ReLU has over sigmoid for deep networks. One should relate to gradients, one to computation.

**Hint**: Think about what happens to gradients when x is large and positive in sigmoid vs ReLU.

**Your Answer**:

[Write your answer here in 2-4 sentences]

### Question 6 (Short Answer)

**Question 6 - Gradient Descent Intuition**

Gradient descent updates weights using: θ_new = θ_old - α × ∇L

Where ∇L is the gradient of the loss.

Explain in simple terms:
1. What does the gradient ∇L represent geometrically?
2. Why do we SUBTRACT it (the negative sign)?
3. What role does α (learning rate) play?

**Hint**: Think of the loss function as a landscape/terrain you're trying to navigate.

**Your Answer**:

[Write your answer here in 2-4 sentences]

### Question 7 (Multiple Choice)

**Question 7 - Loss Function Purpose**

The loss function in deep learning serves to:

A) Measure how wrong the model is, providing a signal for gradient descent
B) Prevent overfitting by penalizing complex models
C) Speed up training by reducing computation
D) Automatically select which features to learn

A) Measure how wrong the model is, providing a signal for gradient descent
B) Prevent overfitting by penalizing complex models
C) Speed up training by reducing computation
D) Automatically select which features to learn

**Hint**: What do we need to compute gradients?

**Your Answer**: [Write your answer here - e.g., 'B']

**Explanation**: [Explain why this is correct]

### Question 8 (Short Answer)

**Question 8 - Feature Learning Reflection**

After running the 'Observing Feature Learning' experiment:

Look at the learned weights in the first layer. These represent the FEATURES the network learned.

Explain: How did the network 'know' which features to learn? What guided it to learn useful features rather than random ones?

**Hint**: The answer involves both the loss function and backpropagation.

**Your Answer**:

[Write your answer here in 2-4 sentences]

### Question 9 (Short Answer)

**Question 9 - Connecting the Concepts**

Integrate all three key insights:

Explain how (1) automatic feature learning, (2) nonlinearity, and (3) gradient descent work TOGETHER to enable deep learning.

Your answer should show how all three are necessary and how they interact.

**Hint**: Think: What would happen if you removed any one of these three components?

**Your Answer**:

[Write your answer here in 2-4 sentences]

### Question 10 (Short Answer)

**Question 10 - Scaling to Real Problems**

ImageNet (image classification) has 1000 classes and ~1.2 million training images. Traditional ML would require human experts to manually design thousands of features.

Explain: Why does deep learning have an advantage that GROWS as the problem gets more complex (more classes, more data)? What breaks down in the traditional approach?

**Hint**: Consider both the human effort required and what happens when you have more data.

**Your Answer**:

[Write your answer here in 2-4 sentences]

---
## Submission

Before submitting:
1. Run all cells to ensure code executes without errors
2. Check that all questions are answered
3. Review your explanations for clarity

**To Submit**:
- File → Download → Download .ipynb
- Submit the notebook file to your course LMS

**Note**: Make sure your name is in the filename (e.g., homework_01_yourname.ipynb)